In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
!pip install git+https://github.com/google-deepmind/recurrentgemma.git
!pip install streamlit

  Cloning https://github.com/google-deepmind/recurrentgemma.git to /tmp/pip-req-build-qctakupz
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/recurrentgemma.git /tmp/pip-req-build-qctakupz
  Resolved https://github.com/google-deepmind/recurrentgemma.git to commit c70d42d9fecfe8b9f27e8c70bd098075c90123ec
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.5 MB/s eta 0:00:00
  Created wheel for recurrentgemma: filename=recurrentgemma-1.0.0-py3-none-any.whl size=87747 sha256=6a049a1d4b5bd2470335dbc69c50559d8a6763ec1c3c02315d2bbfa3c34815ef
  Stored in directory: /tmp/pip-ephem-wheel-cache-1e97jyxd/wheels/88/00/4c/fe9af087e69e42f88b7aa1e14c161a0b0d077eb34f7951bcb7
Successfully built recurrentgemma
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.4.4
    Uninstalling typeguard-4.4

In [ ]:
%%writefile app.py
import streamlit as st
import jax
import flax.serialization as serialization
import sentencepiece as spm
from recurrentgemma import jax as recurrentgemma
import pathlib
import os

# ========================================
# KONFIGURASI MODEL
# ========================================
MODEL_FILE = "/content/drive/MyDrive/Kuliah/Tugas Akhir/Model/skenario_3.msgpack"
TOKENIZER_FILE = "/content/drive/MyDrive/Kuliah/Tugas Akhir/Model/tokenizer.model"
PRESET_VARIANT = "2b"
GENERATION_STEPS = 256
MAX_INPUT_LENGTH = 1024

# ========================================
# TOKENIZER WRAPPER
# ========================================
class GriffinTokenizer:
    def __init__(self, sp_model):
        self._sp_model = sp_model

    def tokenize(self, text, prefix="", suffix="", add_eos=True):
        tokens = [self._sp_model.bos_id()]
        tokens += self._sp_model.EncodeAsIds(prefix + text + suffix)
        if add_eos:
            tokens.append(self._sp_model.eos_id())
        return jax.numpy.array(tokens, dtype=jax.numpy.int32)

    def to_string(self, tokens):
        return self._sp_model.DecodeIds(tokens.tolist())

# ========================================
# FUNGSI MEMUAT MODEL & TOKENIZER
# ========================================
@st.cache_resource
def load_model_and_tokenizer():
    # Di Colab, path seringkali langsung di /content/ atau direktori saat ini
    artifacts_path = pathlib.Path(".").resolve()
    model_path = artifacts_path / MODEL_FILE
    tokenizer_path = artifacts_path / TOKENIZER_FILE

    if not model_path.exists() or not tokenizer_path.exists():
        st.error(f"File tidak ditemukan! Pastikan '{MODEL_FILE}' dan '{TOKENIZER_FILE}' sudah di-upload ke Files (panel kiri).")
        st.stop()

    # Load tokenizer
    sp = spm.SentencePieceProcessor()
    sp.Load(str(tokenizer_path))
    tokenizer = GriffinTokenizer(sp)

    # Buat model
    preset = recurrentgemma.Preset.RECURRENT_GEMMA_2B_V1
    model_config = recurrentgemma.GriffinConfig.from_preset(preset)
    # PENTING: Set dtype ke bfloat16 atau float32 jika perlu, tapi default biasanya oke di GPU
    model = recurrentgemma.Griffin(model_config)

    # Load parameter
    with open(model_path, "rb") as f:
        model_bytes = f.read()

    dummy_tokens = jax.numpy.zeros((1, 1), dtype=jax.numpy.int32)
    dummy_positions = jax.numpy.zeros((1, 1), dtype=jax.numpy.int32)
    key = jax.random.PRNGKey(0)
    initial_params = model.init({'params': key, 'dropout': key}, dummy_tokens, dummy_positions)['params']
    trained_params = serialization.from_bytes(initial_params, model_bytes)

    # Buat sampler
    sampler = recurrentgemma.Sampler(
        model=model,
        params=trained_params,
        vocab=sp
    )

    return sampler, tokenizer

# ========================================
# STREAMLIT UI
# ========================================
st.set_page_config(page_title="📰 News Summarizer", layout="wide")
st.title("📰 Automatic News Summarizer App")
st.markdown("Powered by fine-tuned Recurrent Gemma-2B.")

with st.spinner("Loading model..."):
    sampler, tokenizer = load_model_and_tokenizer()
st.success("Model successfully loaded!")

st.subheader("Enter News Text")
input_text = st.text_area(
    label="News text",
    placeholder="Example: JAKARTA, KOMPAS.com – ...",
    height=300
)

if st.button("✨ Summarize Now", type="primary", use_container_width=True):
    if not input_text.strip():
        st.warning("Masukkan teks yang valid.")
    elif len(input_text.strip()) < 50:
        st.warning("Teks terlalu pendek.")
    else:
        with st.spinner("Meringkas..."):
            prompt = (
                "Ringkas paragraf berikut dalam satu paragraf yang komprehensif dan padat:\n\n"
                f"{input_text.strip()}\n"
            )
            input_tokens = tokenizer.tokenize(prompt, add_eos=False)
            if len(input_tokens) > MAX_INPUT_LENGTH:
                st.info("Teks dipotong agar muat di memori.")
                input_tokens = input_tokens[:MAX_INPUT_LENGTH]
                prompt = tokenizer.to_string(input_tokens)

            output = sampler([prompt], total_generation_steps=GENERATION_STEPS)
            summary = output.text[0].replace("<pad>", "").replace("<eos>", "").strip()

        st.subheader("📄 Summary of Results")
        st.success(summary)

Overwriting app.py


In [ ]:
# 1. Dapatkan IP public Colab Anda (diperlukan untuk password LocalTunnel)
!wget -q -O - ipv4.icanhazip.com

# 2. Jalankan Streamlit di background & expose lewat LocalTunnel
!streamlit run app.py & npx localtunnel --port 8501

35.196.166.251
⠙

⠹⠸⠼⠴⠦⠧
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.30.0.2:8501
  External URL: http://35.196.166.251:8501

y
your url is: https://five-facts-fold.loca.lt
/root/.npm/_npx/75ac80b86e83d4a2/node_modules/localtunnel/bin/lt.js:81
    throw err;
    ^

Error: connection refused: localtunnel.me:32417 (check your firewall settings)
    at Socket.<anonymous> (/root/.npm/_npx/75ac80b86e83d4a2/node_modules/localtunnel/lib/TunnelCluster.js:52:11)
    at Socket.emit (node:events:524:28)
    at emitErrorNT (node:internal/streams/destroy:169:8)
    at emitErrorCloseNT (node:internal/streams/destroy:128:3)
    at process.processTicksAndRejections (node:internal/process/task_queues:82:21)

Node.js v20.19.0
⠙  Stopping...


In [ ]:
# 1. Instal pyngrok
!pip install -q pyngrok

# 2. Masukkan Authtoken Anda (GANTI 'TOKEN_ANDA_DISINI' DENGAN TOKEN DARI WEBSITE NGROK)
from pyngrok import ngrok
import os

# Hentikan tunnel lama jika ada
ngrok.kill()

# Set token
ngrok.set_auth_token("30iofe531fTJDFfo9NvsZ6oRyYE_oNJ3bKiaosSJQ5vpPyXt") # <--- Tempel token di sini

# 3. Jalankan Streamlit di background
import subprocess
process = subprocess.Popen(['streamlit', 'run', 'app.py'])

# 4. Buka Tunnel ke port 8501
public_url = ngrok.connect(8501)
print(f"🚀 Aplikasi Anda berjalan di: {public_url}")

🚀 Aplikasi Anda berjalan di: NgrokTunnel: "https://ccbce5b346a9.ngrok-free.app" -> "http://localhost:8501"
